In [ ]:
import pandas as pd
import numpy as np
from hierarchicalforecast.utils import aggregate
import holidays
import matplotlib.pyplot as plt
import ruptures as rpt
from statsmodels.tsa.seasonal import STL


In [ ]:
sales = pd.read_csv("../eabl parsed csv/sales_products_forecast_ready.csv")
sales


In [ ]:
cm_trimmed = pd.read_excel("../EABL dataset/CustomerMaster_clean_forecast.xlsx")
cm_trimmed

In [ ]:
sales.isna().sum()

In [ ]:
cm_trimmed.isna().sum()

In [ ]:
df = sales.merge(
    cm_trimmed,
    left_on="customer_code",
    right_on="DMS Customer Code",
    how="left"
)



In [ ]:
print(df.shape)
print("unmatched customers:", df["DMS Customer Code"].isna().mean() * 100, "%")

df.to_csv("../eabl parsed csv/hierarchical_forecast_dataset.csv", index=False)

In [ ]:

# Null count and percentage for every column, sorted highest first
null_summary = pd.DataFrame({
    "nulls": df.isna().sum(),
    "pct": df.isna().mean() * 100
}).sort_values("nulls", ascending=False)


print(null_summary[null_summary["nulls"] > 0])


In [ ]:
unmatched = df[df["DMS Customer Code"].isna()]
print(unmatched.shape)
print(unmatched[["customer_code"]])


In [ ]:
df = df[df["DMS Customer Code"].notna()]
print(df.shape)


In [ ]:

# Null count and percentage for every column, sorted highest first
null_summary = pd.DataFrame({
    "nulls": df.isna().sum(),
    "pct": df.isna().mean() * 100
}).sort_values("nulls", ascending=False)

print(null_summary[null_summary["nulls"] > 0])

In [ ]:
for c in ["Division", "Area", "Territory", "Route", "global Channel", "Sub-Channel"]:
    print(c, df[c].nunique(), df[c].isna().sum())


In [ ]:
df.isna().sum()

In [ ]:
df.shape

## 1.4 — Cost of Over- vs. Under-Forecasting: Data-Driven Investigation

**Objective:** Determine whether real cost signals exist in the data to distinguish the cost
of overstocking vs. understocking, per product category/brand — since these costs are not
symmetric (a spoiled case of beer vs. a missed high-margin spirits sale are very different losses).

### Method
Three checks were run against the project data:
1. Return rate by `Category2`, using `quantity_cases` and `returns_cases` from
   `hierarchical_forecast_dataset.csv` — an attempt to measure overstock/spoilage directly.
2. Average price per case by `Category2`, from `product_master_cleaned.csv` — a proxy for the
   cost of a missed sale (understock cost).
3. Real "Expired [Brand]" write-off volume, extracted from `unmatched_products_review.csv`
   (`return_quantity` field) — since these transactions were excluded from the main pipeline.

### Findings

| Check | Result |
|---|---|
| Return rate by category | Beer, RTD, Spirits all show **0%** — `returns_cases` only tracks bottle/crate deposits ("Returnable" = 72.7%), not spoilage. This field cannot be used as an overstock signal for alcohol categories. |
| Price per case | Spirits (~20,502/case) is **~6.4x** the price of Beer (~3,210/case) — a real, usable proxy for understock cost (lost margin per missed sale). |
| Real write-offs | 163 total units expired across 9 SKUs: Guinness (76), White Cap (28), Tusker Ndimu (27), Baileys (22), Gordons Premix (10). |

### Conclusion

- No usable spoilage-cost field exists in the current pipeline — this has to be sourced from
  finance directly.
- Real spoilage is **not confined to Beer**. Baileys and Gordons (both classified under
  `Category2 = Spirits`) had confirmed write-offs — the blanket assumption "Spirits doesn't
  spoil" is too broad and specifically wrong for cream-based/premix products.
- The correct level to apply an overstock/understock cost adjustment is **Brand**, not
  `Category2` — at minimum, treat Guinness, White Cap, Tusker Ndimu, Baileys, and Gordons
  as a separate, higher-overstock-risk group from the rest of Spirits.
- Sample size for the write-off evidence is small (163 units / 16 months) — enough to
  redirect attention to specific brands, not enough to compute a statistically confident
  dollar cost ratio on its own.

### Next steps
- [ ] Bring this exact 5-brand list to finance/supply chain and request real holding,
      spoilage, and stockout-penalty cost figures for these brands specifically.
- [ ] Until real figures are available, use a placeholder lean: order conservatively for
      Beer + the 5 flagged brands; order more generously for the rest of Spirits/RTD.
- [ ] Proceed to Step 1.5 (success metrics + sign-off) in parallel — it does not depend on
      finance's response.


In [ ]:
# ---------- 1. Return rate by category (using returns_cases) ----------

by_cat = df.groupby("Category2").agg(
    total_sold=("quantity_cases", "sum"),
    total_returned=("returns_cases", "sum")
)
by_cat["return_rate_pct"] = by_cat["total_returned"] / by_cat["total_sold"] * 100
print("--- Return rate by category (overstock proxy attempt #1) ---")
print(by_cat.sort_values("return_rate_pct", ascending=False))
print()


In [ ]:
# ---------- 2. Price per case by category (understock proxy) ----------
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")
price_by_cat = pm.groupby("Category2")["FY26_EX_KBL_Price_Per_Case"].agg(["mean", "median", "count"])
print("--- Price per case by category (understock proxy) ---")
print(price_by_cat.sort_values("mean", ascending=False))
print()

# ---------- 3. Real expired/write-off volume by brand (overstock proxy #2) ----------
unmatched = pd.read_csv("../eabl parsed csv/unmatched_products_review.csv")
expired = unmatched[unmatched["product"].str.contains("Expired", case=False, na=False)]
print("--- Expired write-off volume by product (real overstock evidence) ---")
print(expired.groupby("product")["return_quantity"].sum().sort_values(ascending=False))



def adjusted_order_quantity(forecast, brand, category):
    risky_brands = ["Guinness", "White Cap", "Tusker Ndimu", "Baileys", "Gordons"]
    
    if category == "Beer" or brand in risky_brands:
        adjustment = 0.90
    else:
        adjustment = 1.10
    
    return forecast * adjustment

# Example
print(adjusted_order_quantity(100, "Tusker Lager", "Beer"))   # → 90.0 (Beer category, order less)
print(adjusted_order_quantity(100, "Baileys", "Spirits"))     # → 90.0 (flagged brand, order less)
print(adjusted_order_quantity(100, "Smirnoff", "Spirits"))    # → 110.0 (safe Spirits, order more)


In [ ]:
for c in ["Volume Grade", "Archi Type", "Cutsomer Type"]:
    print(f"--- {c} ---")
    print("nulls:", df[c].isna().sum(), f"({df[c].isna().mean()*100:.3f}%)")
    print("distinct values:", df[c].nunique())
    print(df[c].value_counts())
    print()



In [ ]:
print(df["Mode of Payment"].isna().sum())
print(df["Mode of Payment"].value_counts())

In [ ]:
# Does each Sub_Category belong to exactly one Category2?
nesting_check = pm.groupby("Sub_Category")["Category2"].nunique()
print("Sub_Categories spanning more than one Category2:")
print(nesting_check[nesting_check > 1])

# Does each New_Sub_Category2 belong to exactly one Sub_Category?
nesting_check2 = pm.groupby("New_Sub_Category2")["Sub_Category"].nunique()
print("New_Sub_Category2 values spanning more than one Sub_Category:")
print(nesting_check2[nesting_check2 > 1])

print()
print("Cardinality check (should increase going down the tree):")
for c in ["Category2", "Sub_Category", "New_Sub_Category2", "Brand"]:
    print(c, pm[c].nunique())


In [ ]:

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["ds"] = df["visit_date"].dt.to_period("W").dt.start_time


def validate_S(Y_df, S_df, tags, bottom_spec, name):
    print(f"--- Validating {name} ---")
    print("S shape:", S_df.shape)

    # Make sure S_df is indexed by unique_id, not a default row number
    if "unique_id" in S_df.columns:
        S_df = S_df.set_index("unique_id")

    bottom_tag = "/".join(bottom_spec)
    bottom_ids = tags[bottom_tag]

    missing = [i for i in bottom_ids if i not in S_df.index]
    if missing:
        print(f"WARNING: {len(missing)} bottom ids not found in S_df index. Example missing id:", missing[0])
        print("S_df index sample:", list(S_df.index[:5]))
        print()
        return

    S_bottom = S_df.loc[bottom_ids]
    ok_rows = (S_bottom.sum(axis=1) == 1).all()
    ok_cols = (S_bottom.sum(axis=0) == 1).all()
    print("Bottom rows self-map correctly:", ok_rows)
    print("Bottom columns uniquely claimed:", ok_cols)

    if "unique_id" in Y_df.columns:
        # Pick a sample date; missing series that week = 0 sales, not an error
        sample_date = Y_df["ds"].iloc[0]
        y_t_raw = Y_df[Y_df["ds"] == sample_date].set_index("unique_id")["y"]

        b_t = y_t_raw.reindex(bottom_ids, fill_value=0)
        y_t = y_t_raw.reindex(S_df.index, fill_value=0)

        reconstructed = S_df.values @ b_t.values
        identity_holds = np.allclose(reconstructed, y_t.values)
        print("S @ b == y on sample date:", identity_holds)
    print()


# ---------- A. Product ----------
long_product = (
    df.groupby(["Category2", "Sub_Category", "Brand", "Product_Code", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_product = [
    ["Category2"],
    ["Category2", "Sub_Category"],
    ["Category2", "Sub_Category", "Brand"],
    ["Category2", "Sub_Category", "Brand", "Product_Code"],
]
Y_product, S_product, tags_product = aggregate(df=long_product, spec=spec_product)
validate_S(Y_product, S_product, tags_product, spec_product[-1], "Product")

# ---------- B. Geographic — DIAGNOSE FIRST ----------
route_check = df.groupby("Route")[["Division", "Area", "Territory"]].nunique()
problem_routes = route_check[(route_check["Territory"] > 1)]
print("Routes appearing under more than one Territory:")
print(problem_routes)
print()

# Composite key to fix any duplicate Route names across Territories
df["Route_full"] = df["Territory"] + " | " + df["Route"]

long_geo = (
    df.groupby(["Division", "Area", "Territory", "Route_full", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_geo = [
    ["Division"],
    ["Division", "Area"],
    ["Division", "Area", "Territory"],
    ["Division", "Area", "Territory", "Route_full"],
]
Y_geo, S_geo, tags_geo = aggregate(df=long_geo, spec=spec_geo)
validate_S(Y_geo, S_geo, tags_geo, spec_geo[-1], "Geographic (fixed)")

# ---------- C. Channel ----------
long_channel = (
    df.groupby(["global Channel", "Sub-Channel", "segment_grouped", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_channel = [
    ["global Channel"],
    ["global Channel", "Sub-Channel"],
    ["global Channel", "Sub-Channel", "segment_grouped"],
]
Y_channel, S_channel, tags_channel = aggregate(df=long_channel, spec=spec_channel)
validate_S(Y_channel, S_channel, tags_channel, spec_channel[-1], "Channel")

# ---------- D. Grouped — DIAGNOSE FIRST ----------
combo_check = df[["Category2", "segment_grouped"]].drop_duplicates()
print("Distinct Category2 x segment_grouped combinations found:")
print(combo_check.sort_values(["Category2", "segment_grouped"]))
print()

long_grouped = (
    df.groupby(["Category2", "segment_grouped", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_grouped = [
    ["Category2"],
    ["segment_grouped"],
    ["Category2", "segment_grouped"],
]
Y_grouped, S_grouped, tags_grouped = aggregate(df=long_grouped, spec=spec_grouped)
validate_S(Y_grouped, S_grouped, tags_grouped, spec_grouped[-1], "Grouped")



In [ ]:
S_product.to_csv("../eabl parsed csv/S_product.csv")
Y_product.to_csv("../eabl parsed csv/Y_product.csv", index=False)

S_geo.to_csv("../eabl parsed csv/S_geo.csv")
Y_geo.to_csv("../eabl parsed csv/Y_geo.csv", index=False)

S_channel.to_csv("../eabl parsed csv/S_channel.csv")
Y_channel.to_csv("../eabl parsed csv/Y_channel.csv", index=False)

S_grouped.to_csv("../eabl parsed csv/S_grouped.csv")
Y_grouped.to_csv("../eabl parsed csv/Y_grouped.csv", index=False)


In [ ]:
df

## Naive Baseline Forecast (WAPE)

### What this cell does

1. **Load and aggregate to weekly.** Reads the cleaned dataset and converts each date to the week it falls in.
2. **Group by brand and week.** Sums sales so there is exactly **one row per brand per week**.
3. **Build the naive forecast.** `.shift(1)` takes last week's actual sales and uses them as this week's forecast. This is the *naive method*: no model and no math, just the assumption that this week will look like last week.
4. **Measure the error.** Compares the naive forecast against actual sales and rolls the total error into one percentage, the **Weighted Absolute Percentage Error (WAPE)**:

$$
\text{WAPE} = \frac{\sum |\text{Actual} - \text{Forecast}|}{\sum |\text{Actual}|}
$$

   That one number shows how far off the simplest possible approach would have been across the full history.

### Why it matters

This WAPE is the **baseline every real forecasting model has to beat.**

- If "repeat last week" is already ~25% off, a real model needs to land meaningfully **below 25%** to be worth using.
- If it can't, the extra work of hierarchical forecasting doesn't pay off.
- Without a baseline, "our model is good" has no reference point. *Good compared to what?*

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["week"] = df["visit_date"].dt.to_period("W")

# Aggregate sales to Brand x Week
weekly = df.groupby(["Brand", "week"])["quantity_cases"].sum().reset_index()

# The naive forecast for this week = whatever actually happened last week
weekly["naive_forecast"] = weekly.groupby("Brand")["quantity_cases"].shift(1)

# Error = how far off that naive guess was from the real number
weekly["abs_error"] = (weekly["quantity_cases"] - weekly["naive_forecast"]).abs()

# WAPE = total error as a percentage of total actual sales
wape = weekly["abs_error"].sum() / weekly["quantity_cases"].sum() * 100
print("Naive baseline WAPE:", wape)


In [ ]:
kenya_holidays = holidays.Kenya(years=[2025, 2026])
for date, name in sorted(kenya_holidays.items()):
    print(date, name)

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["is_month_end_payday"] = df["visit_date"].dt.day >= (df["visit_date"].dt.days_in_month - 2)

In [ ]:
holiday_df = pd.DataFrame(list(kenya_holidays.items()), columns=["visit_date", "holiday_name"])
holiday_df["visit_date"] = pd.to_datetime(holiday_df["visit_date"])

df = df.merge(holiday_df, on="visit_date", how="left")
df["is_holiday"] = df["holiday_name"].notna()


In [ ]:
df.isna().sum()

In [ ]:
df["holiday_name"] = df["holiday_name"].fillna("normalday")

In [ ]:
df.isna().sum()

In [ ]:
df["holiday_name"].unique()

In [ ]:
df["day_of_week"] = df["visit_date"].dt.dayofweek        # 0=Monday, 6=Sunday
df["is_weekend"] = df["day_of_week"].isin([5, 6])
df["week_of_year"] = df["visit_date"].dt.isocalendar().week
df["month"] = df["visit_date"].dt.month
df["quarter"] = df["visit_date"].dt.quarter
df["is_month_start"] = df["visit_date"].dt.is_month_start
df["is_month_end"] = df["visit_date"].dt.is_month_end

# Cyclical encoding, so the model knows December and January are adjacent
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Days to/from nearest holiday
holiday_dates = pd.to_datetime(sorted(df.loc[df["is_holiday"], "visit_date"].unique()))
def days_to_nearest_holiday(date):
    diffs = (holiday_dates - date).days
    return diffs[np.abs(diffs).argmin()]
df["days_to_nearest_holiday"] = df["visit_date"].apply(days_to_nearest_holiday)


In [ ]:
df

In [ ]:
months = list(range(1, 13))
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

sin_vals = [np.sin(2*np.pi*m/12) for m in months]
cos_vals = [np.cos(2*np.pi*m/12) for m in months]

fig, ax = plt.subplots(figsize=(7,7))

# draw the unit circle for reference
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.sin(theta), np.cos(theta), color='lightgray', linestyle='--', linewidth=1)

# connect months in calendar order, including Dec back to Jan, to show the loop
loop_sin = sin_vals + [sin_vals[0]]
loop_cos = cos_vals + [cos_vals[0]]
ax.plot(loop_sin, loop_cos, color='steelblue', linewidth=1.5, zorder=1)

ax.scatter(sin_vals, cos_vals, color='crimson', s=80, zorder=2)

for m, name, x, y in zip(months, month_names, sin_vals, cos_vals):
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(8,8), fontsize=11, fontweight='bold')

ax.set_xlabel('month_sin')
ax.set_ylabel('month_cos')
ax.set_title('Months placed on a circle using sin/cos\n(Dec connects straight back to Jan)')
ax.set_aspect('equal')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig('month_cycle.png', dpi=150)
plt.show()


In [ ]:
import ruptures as rpt
import matplotlib.pyplot as plt

# ---------- Step 1: Diagnose what the real top-level label looks like ----------
print(Y_product.columns.tolist())
print(Y_product["unique_id"].unique()[:10])
print(list(tags_product.keys()))

# ---------- Step 2: Pick the actual top-level series safely ----------
candidate_labels = ["Total"] + list(Y_product["unique_id"].unique()[:5])

top_level_id = None
for label in candidate_labels:
    subset = Y_product[Y_product["unique_id"] == label]
    if len(subset) > 5:
        top_level_id = label
        break

if top_level_id is None:
    raise ValueError("Could not find a usable top-level series — check the printed unique_id list above.")

print("Using top-level series:", top_level_id)

total_series = Y_product[Y_product["unique_id"] == top_level_id].sort_values("ds")
values = total_series["y"].values
print("Number of data points:", len(values))

# ---------- Step 3: Only run ruptures if there's actually enough data ----------
if len(values) < 10:
    print("Not enough data points to run breakpoint detection reliably.")
else:
    algo = rpt.Pelt(model="rbf").fit(values)
    breakpoints = algo.predict(pen=10)

    print("Detected breakpoints (index positions):", breakpoints)
    print(
        "Corresponding dates:",
        total_series["ds"].iloc[[b - 1 for b in breakpoints[:-1]]].tolist(),
    )

    rpt.display(values, breakpoints)
    plt.title(f"Detected structural breaks — {top_level_id} weekly sales")
    plt.show()

    # Repeat the same check for the other three hierarchies
    for Y_df, name in [(Y_geo, "Geographic"), (Y_channel, "Channel"), (Y_grouped, "Grouped")]:
        candidates = ["Total"] + list(Y_df["unique_id"].unique()[:5])
        for label in candidates:
            s = Y_df[Y_df["unique_id"] == label].sort_values("ds")
            if len(s) > 5:
                vals = s["y"].values
                algo = rpt.Pelt(model="rbf").fit(vals)
                bkps = algo.predict(pen=10)
                print(f"{name} ({label}) breakpoints:", bkps)
                break


In [ ]:
total_by_week = Y_product[Y_product["unique_id"].isin(["Beer", "RTD", "Spirits", "Returnable"])]
total_by_week = total_by_week.groupby("ds")["y"].sum().reset_index().sort_values("ds")

values = total_by_week["y"].values
algo = rpt.Pelt(model="rbf").fit(values)
breakpoints = algo.predict(pen=10)
print("True company-wide breakpoints:", breakpoints)


In [ ]:

# ==================================================================
# PART 1 — What CAN be done: correctly separate "true missing"
# (a data gap) from "confirmed zero" (a real, trustworthy zero sale),
# for EACH product individually, only within its own active window.
# ==================================================================

def classify_series_gaps(df, id_col, date_col, value_col, full_calendar):
    """
    For every bottom-level series (e.g. each Product_Code), compare its
    own active date range against the full calendar and classify each
    period as: true_missing (no row at all) or confirmed_zero/nonzero
    (a real recorded value).
    """
    results = []
    for uid, group in df.groupby(id_col):
        node_start, node_end = group[date_col].min(), group[date_col].max()
        expected = full_calendar[(full_calendar >= node_start) & (full_calendar <= node_end)]
        observed = group.set_index(date_col)[value_col].reindex(expected)

        results.append({
            "unique_id": uid,
            "active_from": node_start,
            "active_to": node_end,
            "true_missing_periods": observed.isna().sum(),
            "confirmed_zero_periods": (observed == 0).sum(),
            "nonzero_periods": (observed > 0).sum(),
        })
    return pd.DataFrame(results)


# Build the weekly date column (Mondays) and a matching calendar (also Mondays)
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["ds"] = df["visit_date"].dt.to_period("W").dt.start_time

# FIX: "W-MON" so the calendar anchors on Mondays, matching df["ds"] exactly
full_calendar = pd.date_range(df["ds"].min(), df["ds"].max(), freq="W-MON")

bottom_df = df.groupby(["Product_Code", "ds"], as_index=False)["quantity_cases"].sum()

gap_classification = classify_series_gaps(
    bottom_df, id_col="Product_Code", date_col="ds",
    value_col="quantity_cases", full_calendar=full_calendar
)

print("Products with the most real data gaps (worth investigating):")
print(gap_classification.sort_values("true_missing_periods", ascending=False).head(10))
print()
print("Overall summary:")
print(gap_classification[["true_missing_periods", "confirmed_zero_periods", "nonzero_periods"]].sum())


In [ ]:


# ==================================================================
# PART 2 — OPTIONAL heuristic: flag WEEKS THAT LOOK LIKE a possible
# stockout (a temporary drop to zero, followed by a recovery). This is
# a guess, not proof — real confirmation needs actual stock data,
# which does not exist in this project.
# ==================================================================

def flag_suspected_stockouts(series, min_run=2, recovery_ratio=0.7):
    """
    series: weekly quantities for ONE product, indexed by date,
    with true gaps already left as NaN (not filled with 0).
    Returns True for weeks that look like a plausible stockout dip.
    """
    vals = series.fillna(0)
    is_zero = (vals == 0)
    flags = pd.Series(False, index=series.index)

    i = 0
    while i < len(vals):
        if is_zero.iloc[i]:
            j = i
            while j < len(vals) and is_zero.iloc[j]:
                j += 1
            before = vals.iloc[max(0, i - 4):i].mean() if i > 0 else np.nan
            after = vals.iloc[j:j + 4].mean() if j < len(vals) else np.nan
            if (j - i) >= min_run and pd.notna(before) and before > 0 and pd.notna(after) and after >= recovery_ratio * before:
                flags.iloc[i:j] = True   # plausible stockout — NOT confirmed
            i = j
        else:
            i += 1
    return flags


# Example: run this on one specific product's weekly series
example_product = bottom_df["Product_Code"].iloc[0]
one_series = (
    bottom_df[bottom_df["Product_Code"] == example_product]
    .set_index("ds")["quantity_cases"]
    .reindex(full_calendar)   # now correctly aligned, Monday-to-Monday
)
suspected = flag_suspected_stockouts(one_series)
print(f"\nSuspected stockout weeks for {example_product}:")
print(suspected[suspected].index.tolist())


In [ ]:


# ==================================================================
# PART 3 — The documentation step: the one thing that IS fully
# actionable today, since real stockout detection needs data that
# does not exist in this project.
# ==================================================================

DATA_SCOPE_NOTE = (
    "NOTE: This forecasting pipeline predicts RECORDED SALES, not true "
    "market demand. No inventory/stockout feed exists in this project as "
    "of {date}. Zero-sale periods are NOT distinguished from stockout-"
    "censored periods. If stock/inventory data becomes available, revisit "
    "this step to implement proper censored-demand estimation before "
    "trusting forecasts for chronically under-supplied SKUs/Routes."
).format(date=pd.Timestamp.today().date())

print("\n" + DATA_SCOPE_NOTE)


In [ ]:
df.isna().sum()

## 4.1 — Missing Data Taxonomy (Structural Zero vs. Stockout vs. True Missing)

**What this section does:**
For every product, compares its actual sales history against a full expected
weekly calendar (within its own active lifetime) to classify each week as
either a real recorded sale, or a gap with no row at all. It also runs a
heuristic that flags weeks which *look like* a possible stockout (a dip in
sales followed by a recovery), and documents the limitations of both checks.

### Key findings
- Across the whole catalog: **2,237 weeks** have no recorded row at all,
  **10,933 weeks** have a real recorded sale, and **0 weeks** show a
  confirmed `0` — because this dataset's parsing process only ever keeps
  rows with a positive quantity, so a genuine "sold nothing" week can't be
  distinguished from "no visit happened" or "out of stock."
- The products with the most gaps are almost entirely low-volume,
  niche/reserve products (e.g. premium whiskies) that naturally sell only
  a handful of times a year — this is expected behavior, not a data error.
- The "suspected stockout" heuristic mostly re-flags these same sparse
  products rather than finding new evidence — for products with very few
  sales, nearly any gap looks like a "dip and recovery" by chance. This is
  a known weakness of the heuristic at low sales volume, not proof of a
  real stockout.

### Significance
This project has **no inventory/stockout data**, so a real zero can never
be fully separated from a hidden stockout with the data currently
available. This section makes that limitation explicit and quantified,
rather than leaving it as a silent assumption. The practical takeaway is
not "go find more stockouts" — it's that the flagged sparse products
should be routed to intermittent-demand-appropriate forecasting methods
(Step 4.4 / 5.6), since standard models will handle them poorly regardless
of the stockout question.

**Scope statement:** this pipeline forecasts *recorded sales*, not true
market demand. Revisit this step if a real stock/inventory feed ever
becomes available.


In [ ]:
stockout_summary

In [ ]:
print(bottom_df["Product_Code"].unique()[:10])
print(bottom_df.shape)
print(stockout_summary.shape)


In [ ]:

# ==================================================================
# VIEW 1 — Histogram: how many "missing" weeks do products typically have?
# ==================================================================
plt.figure(figsize=(8, 5))
plt.hist(gap_classification["true_missing_periods"], bins=30, color="steelblue", edgecolor="white")
plt.xlabel("True missing periods (weeks with no row at all)")
plt.ylabel("Number of products")
plt.title("How many products have how many data gaps?")
plt.tight_layout()
plt.show()


# ==================================================================
# VIEW 2 — Bar chart: the 15 products with the most suspected stockouts
# ==================================================================
top15 = stockout_summary.head(15)

plt.figure(figsize=(9, 6))
plt.barh(top15["Product_Code"].astype(str), top15["count"], color="crimson")
plt.xlabel("Number of suspected stockout weeks")
plt.ylabel("Product Code")
plt.title("Top 15 products by suspected stockout weeks (heuristic, not confirmed)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

all_flags = []
for pid, group in bottom_df.groupby("Product_Code"):
    series = group.set_index("ds")["quantity_cases"].reindex(full_calendar)
    flags = flag_suspected_stockouts(series)
    flagged_weeks = flags[flags].index.tolist()
    if flagged_weeks:
        all_flags.append({
            "Product_Code": pid,
            "suspected_stockout_weeks": flagged_weeks,
            "count": len(flagged_weeks),
        })

stockout_summary = pd.DataFrame(all_flags).sort_values("count", ascending=False)
print(stockout_summary.shape)
print(stockout_summary.head(10))

# ==================================================================
# VIEW 3 — corrected: robust date lookups instead of fragile .get()
# ==================================================================
def plot_product_timeline(product_code, ax):
    series = (
        bottom_df[bottom_df["Product_Code"] == product_code]
        .groupby("ds")["quantity_cases"].sum()   # ensure one value per date, no duplicates
        .reindex(full_calendar)                   # guarantees index is EXACTLY full_calendar
    )
    flags = flag_suspected_stockouts(series).reindex(full_calendar, fill_value=False)

    is_sale = series.fillna(0) > 0
    colors = np.where(
        is_sale, "seagreen",
        np.where(flags, "crimson", "lightgray")
    )

    ax.bar(full_calendar, [1] * len(full_calendar), color=colors, width=6)
    ax.set_yticks([])
    ax.set_title(f"Product {product_code}", loc="left", fontsize=10)


example_products = stockout_summary.head(4)["Product_Code"].tolist()

fig, axes = plt.subplots(len(example_products), 1, figsize=(12, 2 * len(example_products)), sharex=True)
for ax, pid in zip(axes, example_products):
    plot_product_timeline(pid, ax)

plt.xlabel("Week")
fig.suptitle("Week-by-week timeline: green = sold, gray = no row, red = suspected stockout", y=1.02)
plt.tight_layout()
plt.show()


# Demand Classification: Technical Summary

…consistent), **intermittent** (rare, but consistent size when it happens), **erratic** (frequent, but unpredictable size), or **lumpy** (rare *and* unpredictable — the hardest case to forecast).

## Step 2 — Apply it to every individual product

This loops through every `Product_Code`, builds its full weekly history (gaps included as real gaps, not zeros), runs the classification function on it, and collects the results into one table. It then counts how many products fall into each bucket.

This is the part where we expect to find real "intermittent"/"lumpy" products, based on what we already discovered (the Reserve whiskies).

## Step 3 — The "don't just assume" check

This repeats the exact same classification on the bottom level of the other three hierarchies:

- Individual **Routes**
- Individual **Channel–Segment** combinations
- Individual **Category × Segment** cells

It uses the `S` / `Y` / `tags` objects you already built back in Step 2.3.

This tests the expectation that aggregation smooths things out, rather than taking it on faith:

- If these come back overwhelmingly **smooth**, that confirms it.
- If any come back **intermittent** or **lumpy**, that's a genuine exception worth knowing about before you build forecasts for it.

> **Action:** Run this and share all four `value_counts` outputs. That tells us exactly which series need Croston/SBA-style treatment versus standard methods going into F1.

---

# Plain-Language Version

## Step 1 — Sorting products into 4 simple groups

Think of every product as answering two questions:

1. **"How often do you sell?"** Every week, or only once in a while?
2. **"When you do sell, is it always about the same amount, or totally random each time?"**

Combining the answers to those two questions puts every product into one of four groups:

| Group | How often it sells | How much it sells | Difficulty |
|---|---|---|---|
| **Smooth** | Often | Similar amount each time | Easy to predict |
| **Intermittent** | Rarely | Similar amount each time | Moderate |
| **Erratic** | Often | Jumps around unpredictably | Moderate |
| **Lumpy** | Rarely | Unpredictable | Hardest to predict |

## Step 2 — Checking every single product

This runs that same check on every product one at a time and counts how many landed in each of the four groups.

We expect to find several products in the "rare" groups (intermittent/lumpy), like the Reserve whiskies we spotted earlier, which only sell a few times a year.

## Step 3 — Double-checking instead of assuming

Earlier we said that adding lots of products together usually smooths things out. In other words, a whole Route or a whole Channel probably sells steadily, even if some individual products inside it are rare sellers.

Step 3 doesn't just assume that's true. It runs the same 4-group check on **Routes**, **Channels**, and **Category + Segment** combinations too, to make sure nothing rare is hiding at those bigger levels as well.

## Why we do all this

Once we know which group each product, Route, or Channel falls into, we know which forecasting method to use for it. A rare seller needs a different approach than a steady one.

Sorting this out now means the next step (actually building forecasts) starts with the right tool for each situation instead of guessing.

In [ ]:

# ==================================================================
# STEP 1 — Define the classification rule (Syntetos & Boylan, 2005)
# ==================================================================
def classify_demand(series):
    """
    series: weekly quantities for ONE product/node, with true gaps as NaN.
    Returns one of: smooth, intermittent, erratic, lumpy, no_demand.
    """
    nonzero = series.dropna()
    nonzero = nonzero[nonzero > 0]
    if len(nonzero) == 0:
        return "no_demand"

    adi = len(series) / len(nonzero)                    # avg gap between sales
    cv2 = (nonzero.std() / nonzero.mean()) ** 2          # variability of sale size

    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    elif adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    elif adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    else:
        return "lumpy"


In [ ]:
# ==================================================================
# STEP 2 — Run it on Hierarchy A's bottom level (Product_Code)
# — this is where we expect it to matter most
# ==================================================================
results_product = []
for pid, group in bottom_df.groupby("Product_Code"):
    series = group.set_index("ds")["quantity_cases"].reindex(full_calendar)
    results_product.append({"unique_id": pid, "class": classify_demand(series)})

classification_product = pd.DataFrame(results_product)
print("--- Product (SKU level) ---")
print(classification_product["class"].value_counts())
print()

classification_product.to_csv("../eabl parsed csv/classification_product.csv", index=False)

# ==================================================================
# STEP 3 — Spot-check the other three hierarchies' bottom levels too,
# since aggregation *usually* smooths out intermittency, but shouldn't
# just be assumed without checking
# ==================================================================
def classify_bottom_level(Y_df, tags, spec_last_level, full_cal):
    bottom_tag = "/".join(spec_last_level)
    bottom_ids = tags[bottom_tag]
    results = []
    for uid in bottom_ids:
        series = (
            Y_df[Y_df["unique_id"] == uid]
            .set_index("ds")["y"]
            .reindex(full_cal)
        )
        results.append({"unique_id": uid, "class": classify_demand(series)})
    return pd.DataFrame(results)

print("--- Geographic (Route level) ---")
geo_check = classify_bottom_level(Y_geo, tags_geo, spec_geo[-1], full_calendar)
print(geo_check["class"].value_counts())
print()

print("--- Channel level ---")
channel_check = classify_bottom_level(Y_channel, tags_channel, spec_channel[-1], full_calendar)
print(channel_check["class"].value_counts())
print()

print("--- Grouped (Category x Segment) ---")
grouped_check = classify_bottom_level(Y_grouped, tags_grouped, spec_grouped[-1], full_calendar)
print(grouped_check["class"].value_counts())


In [ ]:

# Only Product_Description is genuinely missing from your working dataset —
# pull just that one column from the master file
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")[["Product_Code", "Product_Description"]]
pm["Product_Code"] = pm["Product_Code"].astype(str)

# Brand and Category2 already exist in your working dataset — grab them from there
product_lookup = df[["Product_Code", "Brand", "Category2"]].drop_duplicates()
product_lookup["Product_Code"] = product_lookup["Product_Code"].astype(str)
product_lookup = product_lookup.merge(pm, on="Product_Code", how="left")

classification_product["unique_id"] = classification_product["unique_id"].astype(str)
classification_named = classification_product.merge(
    product_lookup, left_on="unique_id", right_on="Product_Code", how="left"
)

for group_name in ["smooth", "intermittent", "erratic", "lumpy"]:
    subset = classification_named[classification_named["class"] == group_name]
    print(f"\n=== {group_name.upper()} — {len(subset)} products ===")
    print(subset[["Product_Code", "Product_Description", "Brand", "Category2"]].to_string(index=False))



In [ ]:
routing_table = pd.concat([
    classification_product.assign(hierarchy="Product"),
    geo_check.assign(hierarchy="Geographic"),
    channel_check.assign(hierarchy="Channel"),
    grouped_check.assign(hierarchy="Grouped"),
])

routing_table["model_family"] = routing_table["class"].map({
    "smooth": "SeasonalNaive/AutoETS",
    "erratic": "SeasonalNaive/AutoETS",     # still worth standard methods, just noisier
    "intermittent": "Croston/SBA/TSB",
    "lumpy": "Croston/SBA/TSB",
})

print(routing_table.groupby(["hierarchy", "model_family"]).size())


## Forecasting Playbook — Hierarchy Priority & Method Routing

## The Four Hierarchies

| Letter | Name | Structure |
|---|---|---|
| **A** | **Product** | Total → Category2 → Sub_Category → Brand → Product_Code (SKU) |
| **B** | **Geographic** | Total → Division → Area → Territory → Route (using Territory+Route composite key) |
| **C** | **Channel** | Total → global Channel → Sub-Channel → segment_grouped (Bar, Nightclub, Wholesaler, Specialist Store, Event, Lodging, Other) |
| **D** | **Grouped cross-cut** | Category2 × segment_grouped (e.g. "how much Spirits sold specifically through Nightclubs") |


**Decided on:** 2026-09-28, based on ADI/CV² intermittency classification (Step 4.4)

### Priority order to forecast (easiest/most reliable first)
1. **D — Grouped (Category × Segment):** 86% smooth, 0% lumpy/intermittent.
   → Use a single method: SeasonalNaive + AutoETS for the whole hierarchy.
2. **C — Channel:** 59% smooth (small sample, 17 nodes — treat cautiously).
   → Mostly SeasonalNaive/AutoETS; manually check the 6 non-smooth nodes.
3. **B — Geographic (Route):** only 52% smooth — needs a split approach.
   → Smooth/erratic Routes: SeasonalNaive/AutoETS.
   → Intermittent/lumpy Routes: Croston/SBA/TSB.
4. **A — Product (SKU):** only 46% smooth — needs the most care.
   → Smooth/erratic products: SeasonalNaive/AutoETS.
   → Intermittent/lumpy products (e.g. Reserve whiskies: Dalwhinnie, Zacapa,
     Talisker): Croston/SBA/TSB.

### Why this order
Start with the hierarchy needing the least special handling (D) to confirm
the pipeline works end-to-end, before tackling the two hierarchies (A, B)
that need a split-method approach.

### After all four have base forecasts
Reconcile using the S matrices already built in Step 2.3, so Product,
Geographic, Channel, and Total numbers all agree with each other.

### Reference
Classification results and the routing table are saved in
`routing_table.csv` — reload that instead of re-running the classification
from scratch, unless the underlying sales data has materially changed.


In [ ]:
routing_table = pd.concat([
    classification_product.assign(hierarchy="Product"),
    geo_check.assign(hierarchy="Geographic"),
    channel_check.assign(hierarchy="Channel"),
    grouped_check.assign(hierarchy="Grouped"),
])

routing_table["model_family"] = routing_table["class"].map({
    "smooth": "SeasonalNaive/AutoETS",
    "erratic": "SeasonalNaive/AutoETS",
    "intermittent": "Croston/SBA/TSB",
    "lumpy": "Croston/SBA/TSB",
})

routing_table.to_csv("../eabl parsed csv/routing_table.csv", index=False)
print("Saved:", routing_table.shape)


In [ ]:

# Reuse the true company-wide series we already built for the 3.7 structural-break check
total_by_week = Y_product[Y_product["unique_id"].isin(["Beer", "RTD", "Spirits", "Returnable"])]
print(total_by_week)
total_by_week = total_by_week.groupby("ds")["y"].sum().reset_index().sort_values("ds")
total_series = total_by_week.set_index("ds")["y"]

stl = STL(total_series, period=52, robust=True).fit()
seasonal_strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
print("Seasonal strength (0 = none, 1 = very strong):", seasonal_strength)

stl.plot()


In [ ]:
def get_company_total(Y_df, tags, top_level_spec):
    """
    Sums all top-level nodes together to get the true company-wide total
    for any hierarchy, using tags instead of guessing label names.
    """
    top_level_tag = "/".join(top_level_spec)
    top_level_ids = tags[top_level_tag]
    total = (
        Y_df[Y_df["unique_id"].isin(top_level_ids)]
        .groupby("ds")["y"].sum()
        .reset_index()
        .sort_values("ds")
    )
    return total.set_index("ds")["y"]


# Now this works safely for all four, no guessing involved
total_product = get_company_total(Y_product, tags_product, spec_product[0])
total_geo = get_company_total(Y_geo, tags_geo, spec_geo[0])
total_channel = get_company_total(Y_channel, tags_channel, spec_channel[0])
total_grouped = get_company_total(Y_grouped, tags_grouped, spec_grouped[0])


In [ ]:

def check_seasonality(series, name):
    stl = STL(series, period=52, robust=True).fit()
    strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
    print(f"{name} seasonal strength: {strength:.3f}")
    return strength

check_seasonality(total_product, "Product (A)")
check_seasonality(total_geo, "Geographic (B)")
check_seasonality(total_channel, "Channel (C)")
check_seasonality(total_grouped, "Grouped (D)")


In [ ]:

def check_and_plot_seasonality(series, name):
    stl = STL(series, period=52, robust=True).fit()
    strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
    print(f"{name} seasonal strength: {strength:.3f}")

    fig = stl.plot()
    fig.suptitle(f"{name} — Trend / Seasonal / Residual decomposition", y=1.02)
    fig.set_size_inches(10, 8)
    plt.tight_layout()
    plt.show()

    return strength





In [ ]:
strengths = {}
strengths["Product (A)"] = check_and_plot_seasonality(total_product, "Product (A)")


In [ ]:
strengths["Geographic (B)"] = check_and_plot_seasonality(total_geo, "Geographic (B)")


In [ ]:
strengths["Channel (C)"] = check_and_plot_seasonality(total_channel, "Channel (C)")


In [ ]:
strengths["Grouped (D)"] = check_and_plot_seasonality(total_grouped, "Grouped (D)")

print("\nSummary of seasonal strength across hierarchies:")
for name, val in strengths.items():
    print(f"  {name}: {val:.3f}")

In [ ]:
comparison = total_product.rename("product").to_frame()
comparison["geographic"] = total_geo
comparison["channel"] = total_channel
comparison["grouped"] = total_grouped

comparison["all_match"] = (
    (comparison["product"] == comparison["geographic"]) &
    (comparison["product"] == comparison["channel"]) &
    (comparison["product"] == comparison["grouped"])
)

print("Weeks where all four totals match exactly:", comparison["all_match"].sum(), "/", len(comparison))
print(comparison[~comparison["all_match"]].head(10))


In [ ]:

comparison["all_match"] = (
    np.isclose(comparison["product"], comparison["geographic"], rtol=1e-6) &
    np.isclose(comparison["product"], comparison["channel"], rtol=1e-6) &
    np.isclose(comparison["product"], comparison["grouped"], rtol=1e-6)
)

print("Weeks where all four totals match (within tiny rounding tolerance):", comparison["all_match"].sum(), "/", len(comparison))
